# Polars formulation of a causal, vectorized Backtest / Replay boundary

This is a **dummy-data design prototype**, not a replacement for the production
broker or a numbered strategy. Read the intermediate tables before the final P&L.

At each completed boundary, all tickers are rows of one typed table. Historical
market facts and execution-dependent memory are columns. Every calculation over
tickers uses native **Polars expressions, joins, aggregations, or projections**.
No row UDF, `map_elements`, Pandas, NumPy, or per-ticker Python loop is used.
The Python time loop only composes successive causal transitions and collects
their outputs; converting this recurrence to one time-wide query is not claimed.

\[
(M_t,S_t,O_t,A_t;\theta)\mapsto(S_{t+1},O_{t+1},A_{t+1},E_t)
\]

Here `M` is market input, `S` is position / strategy state, `O` is order state,
`A` is account state, and `E` is fill/action evidence. The wide state table below
combines S and O for readability; each row has one ticker and one order slot.


## Declared policy and limits

* One long-only account; integer shares; prices/cash/reservations in integer cents.
  Average cost and realized P&L may be fractional cents; no monetary rounding of
  cash is hidden. No fees, leverage, slippage, corporate actions, or other traders.
* Every market row is a synthetic completed boundary, not an intrabar tape.
  Fresh quotes require `0 <= quote_age_ms <= maximum_quote_age_ms`.
* A BUY limit fills at the fresh ask if ask <= its cap. A SELL market fills at the
  fresh bid. Each ticker has a supplied share-liquidity budget at that boundary.
  There is one order slot per ticker, so no within-ticker order competition.
* Orders submitted at t cannot fill at t. BUY orders expire **before matching**
  when age reaches `entry_ttl_ticks`. Exit orders persist until filled.
* Exits have priority over acquisition: stop, then target. An exit cancels the
  remaining BUY order, releases its reservation, and submits a SELL for held shares.
  Existing SELL orders persist rather than being recreated.
* New entry requests use a common post-fill snapshot. If cash is insufficient,
  all requests are scaled proportionally and rounded down. Rounding residual is
  left as cash. This intentionally differs from sequential greedy admission.
* Initial stop/target are fixed offsets from the proposal ask. Trailing stops
  only rise once the marked price is sufficiently above average cost. These are
  illustrative strategy rules, not the rules of any released strategy.
* Execution phases: expire -> match -> book fills -> propose actions -> cancel
  acquisition for exits -> allocate cash -> submit orders -> snapshot.


In [ ]:
import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(16)
print("Polars", pl.__version__)

# Shape [1, K]: one immutable parameter row, broadcast with cross joins.
policy = pl.DataFrame({
    "maximum_quote_age_ms": [500], "entry_ttl_ticks": [3],
    "requested_shares": [10], "initial_stop_offset_cents": [300],
    "target_offset_cents": [500], "trail_activation_cents": [300],
    "trail_distance_cents": [300],
}).cast(pl.Int64)

# Shape [1, 3]: cash is unreserved total cash; free_cash = cash - reservations.
initial_account = pl.DataFrame({"account_id": ["demo"],
    "cash_cents": [60_000], "realized_pnl_cents": [0.0]})

# Shape [N, K], N=3. AAA starts with four shares at $100; BBB/CCC are flat.
# One explicit order slot per ticker; remaining=0 means the slot is empty.
initial_state = pl.DataFrame({
    "ticker": ["AAA", "BBB", "CCC"], "quantity": [4, 0, 0],
    "book_cost_cents": [40_000.0, 0.0, 0.0],
    "stop_cents": [9_500, 0, 0], "target_cents": [11_000, 0, 0],
    "high_since_fill_cents": [10_000, 0, 0],
    "last_mark_cents": [10_000, 5_000, 4_000],
    "first_fill_tick": [-1, -1, -1],
    "order_side": ["NONE", "NONE", "NONE"],
    "order_remaining": [0, 0, 0], "order_created_tick": [-1, -1, -1],
    "order_limit_cents": [0, 0, 0],
})
print(policy)
print(initial_state)


## Dummy completed market history

AAA falls through its stop. BBB/CCC compete for entry cash at t=0.
CCC subsequently rises through its target while its entry remains unfinished.
BBB later falls through its stop; a stale quote prevents its exit filling at t=4.
The future history is stored together for convenience, but `boundary` receives
only the row set for its current tick. Tick labels are synthetic completed clocks.


In [ ]:
# Shape [T*N, F]: Cartesian product is constructed by Polars, not Python rows.
market = (
    pl.DataFrame({"tick": pl.int_range(0, 7, eager=True)})
    .join(pl.DataFrame({"ticker": ["AAA", "BBB", "CCC"]}), how="cross")
    .with_columns(
        pl.when(pl.col("ticker") == "AAA")
          .then(pl.when(pl.col("tick") < 2).then(10_000)
                .when(pl.col("tick") == 2).then(9_400).otherwise(9_300))
          .when(pl.col("ticker") == "BBB")
          .then(pl.when(pl.col("tick") == 0).then(5_000)
                .when(pl.col("tick") < 3).then(4_900).otherwise(4_600))
          .otherwise(pl.when(pl.col("tick") == 0).then(4_000)
                     .when(pl.col("tick") == 1).then(3_900)
                     .when(pl.col("tick") == 2).then(4_600).otherwise(4_700))
          .cast(pl.Int64).alias("bid_cents"),
        pl.lit(2, dtype=pl.Int64).alias("liquidity_shares"),
        pl.when((pl.col("ticker") == "BBB") & (pl.col("tick") == 4))
          .then(2_000).otherwise(100).cast(pl.Int64).alias("quote_age_ms"),
        ((pl.col("tick") == 0) & pl.col("ticker").is_in(["BBB", "CCC"]))
          .alias("entry_signal"),
    )
    .with_columns((pl.col("bid_cents") + 1).alias("ask_cents"))
    .sort(["tick", "ticker"])
)
assert market.select(pl.struct("tick", "ticker").n_unique()).item() == market.height
print(market)


## Phase 1: match old orders and book fills

`step` is [N, K+F]: join current state to the current completed market boundary,
then broadcast policy. Masks formulate eligibility. `min_horizontal` applies
quantity/liquidity bounds; an account aggregation books all resulting cash flows.
BUY fills add cost; SELL fills remove proportional book cost and realize P&L.


In [ ]:
def match_and_book(state, account, current_market, policy):
    step = (
        state.join(current_market, on="ticker", how="inner", validate="1:1")
        .join(policy, how="cross")
        .with_columns(
            ((pl.col("quote_age_ms") >= 0)
             & (pl.col("quote_age_ms") <= pl.col("maximum_quote_age_ms"))
             & (pl.col("bid_cents") > 0)
             & (pl.col("ask_cents") >= pl.col("bid_cents"))).alias("quote_ready"),
            ((pl.col("order_side") == "BUY") & (pl.col("order_remaining") > 0)
             & (pl.col("tick") - pl.col("order_created_tick")
                >= pl.col("entry_ttl_ticks"))).alias("expired"),
            pl.when(pl.col("quantity") > 0)
              .then(pl.col("book_cost_cents") / pl.col("quantity"))
              .otherwise(0.0).alias("average_cost_before"),
        )
        .with_columns(
            pl.when(pl.col("expired")).then(0)
              .otherwise(pl.col("order_remaining")).alias("live_remaining"),
            pl.when(pl.col("quote_ready")).then(pl.col("bid_cents"))
              .otherwise(pl.col("last_mark_cents")).alias("mark_cents"),
        )
        .with_columns(
            (pl.col("quote_ready") & (pl.col("live_remaining") > 0)
             & (pl.col("tick") > pl.col("order_created_tick"))
             & (((pl.col("order_side") == "BUY")
                 & (pl.col("ask_cents") <= pl.col("order_limit_cents")))
                | (pl.col("order_side") == "SELL"))).alias("fill_eligible"),
            pl.when(pl.col("order_side") == "BUY").then(pl.col("ask_cents"))
              .otherwise(pl.col("bid_cents")).alias("fill_price_cents"),
        )
        .with_columns(
            pl.when(pl.col("fill_eligible"))
              .then(pl.min_horizontal(
                  "live_remaining", "liquidity_shares",
                  pl.when(pl.col("order_side") == "SELL").then(pl.col("quantity"))
                    .otherwise(pl.col("live_remaining"))))
              .otherwise(0).cast(pl.Int64).alias("fill_quantity"),
        )
        .with_columns(
            (pl.col("fill_quantity")
             * pl.when(pl.col("order_side") == "BUY").then(1).otherwise(-1))
              .alias("quantity_delta"),
            (pl.col("fill_quantity") * pl.col("fill_price_cents")
             * pl.when(pl.col("order_side") == "BUY").then(-1).otherwise(1))
              .alias("cash_delta_cents"),
            pl.when(pl.col("order_side") == "SELL")
              .then(pl.col("fill_quantity")
                    * (pl.col("fill_price_cents") - pl.col("average_cost_before")))
              .otherwise(0.0).alias("realized_delta_cents"),
        )
        .with_columns(
            (pl.col("quantity") + pl.col("quantity_delta")).alias("post_quantity"),
            (pl.col("live_remaining") - pl.col("fill_quantity")).alias("post_remaining"),
            (pl.col("book_cost_cents")
             + pl.when(pl.col("order_side") == "BUY")
               .then(pl.col("fill_quantity") * pl.col("fill_price_cents"))
               .otherwise(-pl.col("fill_quantity") * pl.col("average_cost_before")))
              .alias("post_book_cost_cents"),
            pl.when((pl.col("quantity") == 0) & (pl.col("quantity_delta") > 0))
              .then(pl.col("tick")).otherwise(pl.col("first_fill_tick"))
              .alias("post_first_fill_tick"),
        )
    )
    # Shape [N, evidence_fields]: zero fills remain visible with eligibility flags.
    fills = step.select("tick", "ticker", "order_side", "order_created_tick",
                        "expired", "quote_ready", "fill_eligible", "fill_quantity",
                        "fill_price_cents", "quantity_delta", "cash_delta_cents",
                        "realized_delta_cents", "post_remaining")
    totals = step.select(pl.col("cash_delta_cents").sum(),
                         pl.col("realized_delta_cents").sum())
    account = account.join(totals, how="cross").select(
        "account_id", (pl.col("cash_cents") + pl.col("cash_delta_cents")).alias("cash_cents"),
        (pl.col("realized_pnl_cents") + pl.col("realized_delta_cents"))
          .alias("realized_pnl_cents"))
    return step, account, fills


## Phase 2: formulate actions as masks and typed columns

Actions are proposals, not fills. All tickers read the same post-fill snapshot.
Held positions with a pending SELL keep that order. Otherwise stop/target exits
can cancel a remaining BUY. A fresh flat ticker with no order can request entry.
Missing quotes yield WAIT/HOLD/KEEP_ORDER without invented observations.


In [ ]:
def formulate_actions(step):
    return (
        step.with_columns(
            pl.when(pl.col("post_quantity") > 0)
              .then(pl.col("post_book_cost_cents") / pl.col("post_quantity"))
              .otherwise(0.0).alias("post_average_cost"),
            ((pl.col("order_side") == "SELL") & (pl.col("post_remaining") > 0))
              .alias("pending_exit"),
        )
        .with_columns(
            pl.when((pl.col("post_quantity") > 0) & pl.col("quote_ready")
                    & (pl.col("bid_cents") >= pl.col("post_average_cost")
                       + pl.col("trail_activation_cents")))
              .then(pl.max_horizontal("stop_cents",
                                     pl.col("bid_cents") - pl.col("trail_distance_cents")))
              .otherwise(pl.col("stop_cents")).alias("proposed_stop_cents"),
        )
        .with_columns(
            pl.when((pl.col("post_quantity") > 0) & ~pl.col("pending_exit")
                    & pl.col("quote_ready")
                    & (pl.col("bid_cents") <= pl.col("proposed_stop_cents")))
              .then(pl.lit("EXIT_STOP"))
              .when((pl.col("post_quantity") > 0) & ~pl.col("pending_exit")
                    & pl.col("quote_ready")
                    & (pl.col("bid_cents") >= pl.col("target_cents")))
              .then(pl.lit("EXIT_TARGET"))
              .when(pl.col("post_remaining") > 0).then(pl.lit("KEEP_ORDER"))
              .when(pl.col("post_quantity") > 0).then(pl.lit("HOLD"))
              .when(pl.col("quote_ready") & pl.col("entry_signal"))
              .then(pl.lit("ENTER"))
              .otherwise(pl.lit("WAIT")).alias("action"),
        )
        .with_columns(
            pl.col("action").is_in(["EXIT_STOP", "EXIT_TARGET"]).alias("exit_requested"),
            pl.when(pl.col("action") == "ENTER").then(pl.col("requested_shares"))
              .otherwise(0).alias("requested_quantity"),
        )
        .with_columns(
            pl.when(pl.col("exit_requested")).then(0)
              .otherwise(pl.col("post_remaining")).alias("retained_remaining"),
            (pl.col("requested_quantity") * pl.col("ask_cents"))
              .alias("requested_notional_cents"),
        )
    )


## Phase 3: simultaneous allocation and explicit state commit

Existing BUY reservations use remaining shares * original limit, not today's ask.
Exit cancellation releases those reservations before allocation. For requests i:

`scale = min(1, free_cash / sum(request_notional))`

`approved_shares_i = floor(requested_shares_i * scale)`

Zero approvals are recorded, not dropped. No residual cash is greedily reassigned.
Cash changes only on actual fills; reservations constrain new admissions.


In [ ]:
def allocate_and_commit(proposals, account):
    totals = proposals.select(
        pl.when(pl.col("order_side") == "BUY")
          .then(pl.col("retained_remaining") * pl.col("order_limit_cents"))
          .otherwise(0).sum().alias("retained_reservations_cents"),
        pl.col("requested_notional_cents").sum().alias("total_requested_cents"),
    )
    budget = account.join(totals, how="cross").with_columns(
        (pl.col("cash_cents") - pl.col("retained_reservations_cents"))
          .alias("free_cash_before_admission_cents")
    ).with_columns(
        pl.when(pl.col("total_requested_cents") > 0)
          .then(pl.min_horizontal(pl.lit(1.0),
                pl.col("free_cash_before_admission_cents") / pl.col("total_requested_cents")))
          .otherwise(0.0).alias("allocation_scale")
    )
    decisions = proposals.join(budget, how="cross").with_columns(
        # Integer division implements floor(request * scale) without float rounding.
        pl.when(pl.col("total_requested_cents") > 0)
          .then(pl.min_horizontal("requested_quantity",
                (pl.col("requested_quantity") * pl.col("free_cash_before_admission_cents"))
                // pl.col("total_requested_cents")))
          .otherwise(0).cast(pl.Int64).alias("approved_quantity")
    ).with_columns(
        pl.when(pl.col("exit_requested")).then(pl.lit("SELL"))
          .when(pl.col("approved_quantity") > 0).then(pl.lit("BUY"))
          .when(pl.col("retained_remaining") > 0).then(pl.col("order_side"))
          .otherwise(pl.lit("NONE")).alias("next_order_side"),
        pl.when(pl.col("exit_requested")).then(pl.col("post_quantity"))
          .when(pl.col("approved_quantity") > 0).then(pl.col("approved_quantity"))
          .otherwise(pl.col("retained_remaining")).alias("next_order_remaining"),
        ((pl.col("approved_quantity") > 0) | pl.col("exit_requested"))
          .alias("new_order"),
    )
    # Project exactly the state schema: a new immutable table replaces the old one.
    next_state = decisions.select(
        "ticker", pl.col("post_quantity").alias("quantity"),
        pl.when(pl.col("post_quantity") > 0).then(pl.col("post_book_cost_cents"))
          .otherwise(0.0).alias("book_cost_cents"),
        pl.when(pl.col("approved_quantity") > 0)
          .then(pl.col("ask_cents") - pl.col("initial_stop_offset_cents"))
          .when((pl.col("post_quantity") > 0) | (pl.col("next_order_side") == "BUY"))
          .then(pl.col("proposed_stop_cents"))
          .otherwise(0).alias("stop_cents"),
        pl.when(pl.col("approved_quantity") > 0)
          .then(pl.col("ask_cents") + pl.col("target_offset_cents"))
          .when((pl.col("post_quantity") > 0) | (pl.col("next_order_side") == "BUY"))
          .then(pl.col("target_cents"))
          .otherwise(0).alias("target_cents"),
        pl.when(pl.col("post_quantity") == 0).then(0)
          .when((pl.col("quantity") == 0) & (pl.col("quantity_delta") > 0))
          .then(pl.col("mark_cents"))
          .otherwise(pl.max_horizontal("high_since_fill_cents", "mark_cents"))
          .alias("high_since_fill_cents"),
        pl.col("mark_cents").alias("last_mark_cents"),
        pl.when(pl.col("post_quantity") > 0).then(pl.col("post_first_fill_tick"))
          .otherwise(-1).alias("first_fill_tick"),
        pl.col("next_order_side").alias("order_side"),
        pl.col("next_order_remaining").alias("order_remaining"),
        pl.when(pl.col("new_order")).then(pl.col("tick"))
          .when(pl.col("next_order_remaining") > 0).then(pl.col("order_created_tick"))
          .otherwise(-1).alias("order_created_tick"),
        pl.when(pl.col("approved_quantity") > 0).then(pl.col("ask_cents"))
          .when(pl.col("next_order_side") == "BUY").then(pl.col("order_limit_cents"))
          .otherwise(0).alias("order_limit_cents"),
    ).sort("ticker")
    summary = next_state.select(
        pl.when(pl.col("order_side") == "BUY")
          .then(pl.col("order_remaining") * pl.col("order_limit_cents"))
          .otherwise(0).sum().alias("reserved_cents"),
        (pl.col("quantity") * pl.col("last_mark_cents")).sum().alias("market_value_cents"),
        pl.col("book_cost_cents").sum().alias("total_book_cost_cents"),
    )
    snapshot = account.join(summary, how="cross").with_columns(
        (pl.col("cash_cents") - pl.col("reserved_cents")).alias("free_cash_cents"),
        (pl.col("cash_cents") + pl.col("market_value_cents")).alias("equity_cents"),
        (pl.col("market_value_cents") - pl.col("total_book_cost_cents"))
          .alias("unrealized_pnl_cents"),
    )
    actions = decisions.select("tick", "ticker", "quote_ready", "post_quantity",
        "action", "proposed_stop_cents", "requested_quantity", "requested_notional_cents",
        "free_cash_before_admission_cents", "allocation_scale", "approved_quantity",
        "exit_requested", "next_order_side", "next_order_remaining")
    return next_state, snapshot, actions

def boundary(state, account, current_market, policy):
    # These scalar assertions check contracts; they do not implement trading rules.
    assert current_market.select(pl.col("tick").n_unique()).item() == 1
    assert current_market.height == state.height
    assert current_market.select(pl.col("ticker").n_unique()).item() == state.height
    assert state.select(pl.col("ticker").n_unique()).item() == state.height
    assert state.join(current_market, on="ticker", how="anti").is_empty()
    assert current_market.select((pl.col("liquidity_shares") >= 0).all()).item()
    step, post_fill_account, fills = match_and_book(state, account, current_market, policy)
    proposals = formulate_actions(step)
    next_state, snapshot, actions = allocate_and_commit(proposals, post_fill_account)
    assert snapshot.select((pl.col("free_cash_cents") >= 0).all()).item()
    assert next_state.select(((pl.col("quantity") >= 0)
                             & (pl.col("order_remaining") >= 0)).all()).item()
    return next_state, post_fill_account, fills, actions, snapshot


## Inspect one boundary before running the recurrence

At t=0 the two requests exceed $600 cash. Both get six shares, and the unspent
rounding residual remains available. There are no immediate fills. The next cell
shows requests, approved commands, and committed orders separately.


In [ ]:
state_0, account_0, fills_0, actions_0, snapshot_0 = boundary(
    initial_state, initial_account, market.filter(pl.col("tick") == 0), policy)
print("Action proposals and allocation")
print(actions_0)
print("Committed state / working orders")
print(state_0)
print("Account snapshot")
print(snapshot_0)


## Sequential time, vectorized tickers

The only data-processing loop below advances completed boundaries. It never
iterates over tickers/orders. Each call processes all N rows using the same
expression graph. Histories are outputs for inspection; they do not feed future
values back into earlier decisions. Replay would pace these same transitions.


In [ ]:
def run_simulation(market_history):
    state, account = initial_state, initial_account
    fill_batches, action_batches, state_batches, account_batches = [], [], [], []
    # Clock orchestration only; every calculation inside boundary is Polars-native.
    for current_market in market_history.sort(["tick", "ticker"]).partition_by("tick", maintain_order=True):
        state, account, fills, actions, snapshot = boundary(state, account, current_market, policy)
        clock = current_market.select(pl.col("tick").first())
        fill_batches.append(fills)
        action_batches.append(actions)
        state_batches.append(state.join(clock, how="cross"))
        account_batches.append(snapshot.join(clock, how="cross"))
    return {
        "fills": pl.concat(fill_batches), "actions": pl.concat(action_batches),
        "states": pl.concat(state_batches), "accounts": pl.concat(account_batches),
    }

result = run_simulation(market)
print("Executed fills")
print(result["fills"].filter(pl.col("fill_quantity") > 0))
print("Nontrivial decisions")
print(result["actions"].filter(~pl.col("action").is_in(["HOLD", "WAIT"])))
print("Account history (cents)")
print(result["accounts"])


## Meaningful checks: accounting, partial fills, reservations and causality

These checks compare independent accounting identities and scenario outcomes.
Future-tail independence reruns the same simulation after changing prices only
at t>=4. States, actions, fills and accounts through t=3 must remain identical.
This is design validation on dummy data, not production execution parity.


In [ ]:
# Cash is conserved against actual fills, not requested or approved order size.
cash_check = result["accounts"].join(
    result["fills"].group_by("tick").agg(pl.col("cash_delta_cents").sum())
      .sort("tick").with_columns(pl.col("cash_delta_cents").cum_sum().alias("cash_change")),
    on="tick", validate="1:1")
assert cash_check.select((pl.col("cash_cents") == 60_000 + pl.col("cash_change")).all()).item()

quantity_check = result["fills"].sort(["ticker", "tick"]).with_columns(
    pl.col("quantity_delta").cum_sum().over("ticker").alias("cumulative_delta")
).join(initial_state.select("ticker", pl.col("quantity").alias("initial_quantity")), on="ticker")
quantity_check = quantity_check.join(result["states"].select("tick", "ticker", "quantity"),
                                    on=["tick", "ticker"], validate="1:1")
assert quantity_check.select((pl.col("quantity") == pl.col("initial_quantity")
                             + pl.col("cumulative_delta")).all()).item()
assert result["fills"].filter(pl.col("fill_quantity") > 0).select(
    (pl.col("tick") > pl.col("order_created_tick")).all()).item()
assert result["accounts"].select((pl.col("reserved_cents") <= pl.col("cash_cents")).all()).item()
assert actions_0.filter(pl.col("action") == "ENTER").select(
    (pl.col("approved_quantity") == 6).all()).item()
assert result["states"].filter((pl.col("tick") == 1) & (pl.col("ticker") == "BBB")).select(
    ((pl.col("quantity") == 2) & (pl.col("order_remaining") == 4)).all()).item()
assert result["actions"].filter((pl.col("tick") == 2) & (pl.col("ticker") == "CCC")).select(
    ((pl.col("action") == "EXIT_TARGET") & (pl.col("next_order_side") == "SELL")
     & (pl.col("next_order_remaining") == 2)).all()).item()
assert result["fills"].filter((pl.col("tick") == 3) & (pl.col("ticker") == "BBB")).select(
    pl.col("expired").all()).item()
assert result["fills"].filter((pl.col("tick") == 4) & (pl.col("ticker") == "BBB")).select(
    ((pl.col("fill_quantity") == 0) & ~pl.col("quote_ready")).all()).item()
assert result["states"].filter(pl.col("tick") == 6).select(
    ((pl.col("quantity") == 0) & (pl.col("order_remaining") == 0)).all()).item()
assert result["accounts"].select(
    ((pl.col("equity_cents") - 100_000 - pl.col("realized_pnl_cents")
      - pl.col("unrealized_pnl_cents")).abs() < 1e-6).all()).item()

changed_future = market.with_columns(
    pl.when(pl.col("tick") >= 4).then(pl.col("bid_cents") + 2_000)
      .otherwise(pl.col("bid_cents")).alias("bid_cents"),
    pl.when(pl.col("tick") >= 4).then(pl.col("ask_cents") + 2_000)
      .otherwise(pl.col("ask_cents")).alias("ask_cents"),
)
counterfactual = run_simulation(changed_future)
assert result["states"].filter(pl.col("tick") <= 3).equals(counterfactual["states"].filter(pl.col("tick") <= 3))
assert result["actions"].filter(pl.col("tick") <= 3).equals(counterfactual["actions"].filter(pl.col("tick") <= 3))
assert result["fills"].filter(pl.col("tick") <= 3).equals(counterfactual["fills"].filter(pl.col("tick") <= 3))
assert result["accounts"].filter(pl.col("tick") <= 3).equals(counterfactual["accounts"].filter(pl.col("tick") <= 3))
assert result["states"].equals(run_simulation(market.reverse())["states"])

# A BUY that cannot fill must retain its protection specification while flat.
delayed_market = market.with_columns(
    pl.when((pl.col("ticker") == "BBB") & pl.col("tick").is_in([1, 2]))
      .then(2_000).otherwise(pl.col("quote_age_ms")).alias("quote_age_ms"))
delayed = run_simulation(delayed_market)
assert delayed["states"].filter((pl.col("ticker") == "BBB") & pl.col("tick").is_in([1, 2])).select(
    ((pl.col("quantity") == 0) & (pl.col("order_side") == "BUY")
     & (pl.col("stop_cents") == 4_701) & (pl.col("target_cents") == 5_501)).all()).item()
print("PASS: cash/quantity conservation, delayed fills, bounded reservations, partial fills,")
print("exit cancellation, expiry, stale-quote deferral, terminal flattening, P&L, future-tail independence,")
print("input-order independence and protection retention before first fill.")


## What to review next

This formulation demonstrates stateful decisions as columnar expressions. It
does not assert that all strategies can share these exact columns or policies.
To extend it: add account/ticker keys; normalize multiple orders into an order
table; define segmented liquidity and capital allocation; pin action precedence;
and specify actual execution clocks and producer dependencies. Multiple orders
competing for liquidity require a declared resolver, not independent fills.

Production work must preserve old numbered releases, certified input coverage,
Portfolio/OMS authority, normalized durability, and causal recovery. The simplified
pro-rata policy must not silently replace the existing ordered policy.
